# CN7 재시도: 모델2~3개 진단 비교
사전에 정한 작은 조합의 라벨 탐지와 비라벨 경보 비율. 새 독립 검증이나 안쪽 자동 선택 모델이라고 주장하지 않는다.


In [1]:
from pathlib import Path
import itertools,json
import pandas as pd,numpy as np
from sklearn.metrics import confusion_matrix
ROOT=Path.cwd();t=ROOT/"reports/tables"
base=pd.read_csv(t/"cn7_reference_snapshot/cn7_165_gas_supervised_complementary_predictions.csv",parse_dates=["TimeStamp"]).sort_values("TimeStamp").reset_index(drop=True)
times=base.TimeStamp
y=base.actual.to_numpy(int);eligible=base.eligible.to_numpy(bool)
assert eligible.sum()==605 and y[eligible].sum()==13
def align(file):
 a=pd.read_csv(t/file,parse_dates=["TimeStamp"])
 return a.set_index("TimeStamp").loc[times]
def head(file,model,rule):
 a=pd.read_csv(t/file,parse_dates=["TimeStamp"])
 a=a[(a.model==model)&(a.rule==rule)].set_index("TimeStamp").loc[times]
 return a.prediction.to_numpy(bool)
confirm=align("cn7_177_confirmed_expert_predictions.csv")
heads={
 "relation_ET":base.baseline.to_numpy(bool),
 "joint_LR":base.joint.to_numpy(bool)&base.pressure_direction_confirm.to_numpy(bool),
 "load_LR":base.load.to_numpy(bool)&base.load_direction_confirm.to_numpy(bool),
 "persistent_ET":head("cn7_170_persistence_predictions.csv","persistent_abs4_ET","normal95"),
 "transfer_LR":head("cn7_173_transfer_predictions.csv","mechanism_joint8_LR","inner_normal0.99"),
 "local_flow_RF":confirm.localRF_fast.to_numpy(bool),
 "abs_flow_LR":confirm.absoluteLR_meter.to_numpy(bool)}
u=pd.read_csv(t/"cn7_179_unlabeled_predictions.csv",parse_dates=["TimeStamp"])
mask=u.eligible_unlabeled.to_numpy(bool)
assert mask.sum()==27768
# Definitions fixed BEFORE seeing this comparison's scores.
fixed={
 "old_seven":tuple(heads),
 "trees_only3":("relation_ET","persistent_ET","local_flow_RF"),
 "persistent_RF2":("persistent_ET","local_flow_RF"),
 "persistent_RF_absLR3":("persistent_ET","local_flow_RF","abs_flow_LR"),
 "original_three":("relation_ET","joint_LR","load_LR")}
def stats(a,b):
 tn,fp,fn,tp=confusion_matrix(a,b,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),
 precision=float(tp/(tp+fp)) if tp+fp else 0,
 recall=float(tp/(tp+fn)),
 F1=float(2*tp/(2*tp+fp+fn)) if tp else 0)
def evaluate(names,label):
 pred=np.any(np.column_stack([heads[n] for n in names]),axis=1)
 upred=u[list(names)].any(axis=1).to_numpy()
 row=dict(policy=label,models="+".join(names),model_count=len(names),
 **stats(y[eligible],pred[eligible]),unlabeled_eligible=int(mask.sum()),
 unlabeled_candidates=int(upred[mask].sum()),
 unlabeled_alarm_percent=100*upred[mask].mean())
 return row,pred,upred
rows=[]
for count in [1,2,3]:
 for names in itertools.combinations(heads,count):
  row,_,_=evaluate(names,"exploratory")
  rows.append(row)
explore=pd.DataFrame(rows)
fixedrows=[];predictions=base[["TimeStamp","actual","eligible"]].copy()
for label,names in fixed.items():
 row,p,up=evaluate(names,label);fixedrows.append(row)
 predictions[label]=p.astype(int)
 u[label]=np.where(mask,up.astype(int),np.nan)
fm=pd.DataFrame(fixedrows)
explore.to_csv(t/"cn7_180_small_ensemble_exploratory.csv",index=False)
fm.to_csv(t/"cn7_180_small_ensemble_fixed_metrics.csv",index=False)
predictions.to_csv(t/"cn7_180_small_ensemble_labeled_predictions.csv",index=False)
u[["TimeStamp","history_run","known_label","eligible_unlabeled","in_reference_support"]+list(fixed)].to_csv(t/"cn7_180_small_ensemble_unlabeled_predictions.csv",index=False)
print("FIXED COMPARISONS\n",fm.to_string(index=False))
print("\nLOW ALARM EXPLORATORY (NOT SELECTED/VALIDATED)\n",explore[explore.unlabeled_alarm_percent<=2].sort_values(["F1","FP"],ascending=[False,True]).head(10).to_string(index=False))
for name in fixed:
 if name=="old_seven":continue
 print("\n",name,"gas",predictions[predictions.actual.eq(1)&predictions[name].eq(1)].TimeStamp.dt.strftime("%H:%M:%S").tolist())
# Frozen small candidates are diagnostic comparisons, not chosen by maximizing outer labels.
# Do not fit a stacking model on these outer OOF predictions: their generating fits can contain
# labels from another fold, which would leak when meta-training within that fold.
lines=["# CN7 재시도: 작은 조합과 비라벨 경보 함께 보기","",
 "7개/비라벨27.96% 버전은 실사용 후보에서 보류했다. 이번에는 기존 피처/학습 기준을 바꾸지 않고 모델을 제외한 조합을 비교했다.",
 "RF/ET 위주의2~3개 조합과 저경보LR1개를 추가한 조합을 비교 실행 전에 정했다.",
 "정상592/가스13의605샷, 기존 시간블록/purge10 예측을 같은Timestamp로 연결한다.",
 "비라벨은179에서 전체 재학습한 모델 출력으로 비교한다. 원래 구간별 검증 모델과 다르고, 새 분포 독립검증이 아니다.",
 "임계값을 비라벨 경보비율에 맞춰 조절하거나 비라벨을 정상으로 라벨링하지 않았다.","",
 "## 사전에 정의한 조합",fm.to_string(index=False),"",
 "## 해석",
 "경보율이 낮다고 정확한 모델은 아니다. 놓친 가스 수도 반드시 함께 본다.",
 "작은 조합에서 정상 경보가 줄어드는지와 라벨 탐지를 유지하는지 살펴본 진단이다. 새로운 학습 내부 선택 검증을 했다는 뜻은 아니다.",
 "모든1~3개 조합의 표는 탐색 참고용이며 평가 라벨을 본 뒤 최고를 골랐다고 독립 성능으로 주장하면 안 된다.",
 "구간별OOF 예측을 그대로 stacking 학습에 쓰면 다른구간 라벨 유입 가능성이 있으므로 이번에는 메타모델을 학습하지 않았다.","",
 "## 다음 우선순위",
 "작은 조합을 기준으로 새 피처를 안쪽에서 선택하는 검증이 필요하다.",
 "기존 전체재학습 임계값을 구간별 기준 중앙값으로 연결한 가정, 운전모드 매칭, 희소/밀집 이력 차이는 여전히 별도 해결 대상이다.",
 "따라서 경보율만 낮아졌다고 실사용 가능으로 결론내리지 않는다."]
(ROOT/"reports/cn7_180_small_ensemble_retry.md").write_text("\n".join(lines),encoding="utf-8")


FIXED COMPARISONS
               policy                                                                           models  model_count  TP  FP  FN  TN  precision   recall       F1  unlabeled_eligible  unlabeled_candidates  unlabeled_alarm_percent
           old_seven relation_ET+joint_LR+load_LR+persistent_ET+transfer_LR+local_flow_RF+abs_flow_LR            7  10   5   3 587   0.666667 0.769231 0.714286               27768                  7765                27.963843
         trees_only3                                          relation_ET+persistent_ET+local_flow_RF            3   5   3   8 589   0.625000 0.384615 0.476190               27768                    25                 0.090032
      persistent_RF2                                                      persistent_ET+local_flow_RF            2   5   2   8 590   0.714286 0.384615 0.500000               27768                    25                 0.090032
persistent_RF_absLR3                                          persistent_

2116